# Dropout Optimizations

The old implementation is shown below: 

In [ ]:
try:
    import cupy as cp
    xp = cp
except (ModuleNotFoundError, ImportError):
    import numpy as np
    xp = np 

class Dropout:
    def __init__(self, rate):
        #We write rate as the success rate. The dropout rate will then be 
        self.rate = 1 - rate
    
    def forward(self, inputs, training):
        #were gonna save the inputs and the binary mask
        self.inputs = inputs
        if not training:
            self.output = inputs.copy()
            return self.output
        self.binary_mask = xp.random.binomial(1, self.rate, size = inputs.shape) \
                        / self.rate
        self.output = self.binary_mask * self.inputs

        return self.output
    
    def backward(self, dvalues):
        self.dinputs = dvalues * self.binary_mask 

## Can we Really make any Improvements for Dropout? 

In general, with GPU programming there are two types of bounds to account for. 

* **Compute Bound**: Where the amount of FLOPs(floating point operations) of a compute core are saturated, meaning data has to wait for a cores thread to complete its task before moving on. 
* **Memory Bound**: Where the compute cores are waiting for data to be fetched from GPU VRAM. In this case, while GPUs benefit from holding information inside L1 and L2 cache, and perform calculations using the GPU registers. Lower level memory types can't account for slow VRAM, meaning compute cores idle waiting for new work. 

Our solution to memory bound, or more specifically memory bandwidth problems come from **kernel fusion** (`@cp.fuse()`, `cp.ElementwiseKernel`):  
  
We can circumvent this by avoiding unnecessary kernel operations, oftentimes combining operations (otherwise combining kernels) into a single or custom kernel that performs multiple operations at the same time. 
* A fused kernel will have data read from VRAM once, with operations being performed on the fly with the faster GPU registers and cache. Finally, we create a final tensor **once**, and write this back onto GPU VRAM. 

In our old implementation, we required 4 seperate kernel operations. Assume N elements means a 4D tensor:

    Step 1: tmp1 = xp.random.binomial(1, ...)                   ──► Launches Kernel 1 (Allocates N elements for tmp1)  
    Step 2: self.binary_mask = tmp1 / self.rate                 ──► Launches Kernel 2 (Allocates N elements for binary_mask)  
    Step 3: self.output = self.binary_mask * self.inputs        ──► Launches Kernel 3 (Allocates N elements for output)
This means we read from VRAM 3 times, and write to VRAM 3 times. We'll rewrite this pass such that we read from VRAM once, and write to VRAM once. We will be implementing the following:

    Step 1: self.output = _fused_dropout(inputs, self.rate)     ──► Launches Kernel 1 (Allocates N elements ONLY for output)
